<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-04-rag/lesson-4.4-search-grounding/notebooks/GCP_Capstone_4.4_Search_Grounding.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4.4 Vertex AI Search & Google Search Grounding
**Netsetos GenAI Engineering — GCP Capstone**

Enterprise search over your data + live web grounding. Two GCP-unique RAG approaches.


## Setup


In [ ]:
!pip install -q google-genai google-cloud-discoveryengine
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE
LOCATION = 'global'

from google import genai
from google.genai import types
from google.cloud import discoveryengine_v1 as discoveryengine

client = genai.Client(enterprise=True, project=PROJECT_ID, location='global')  # Gemini 3.x generation: global


## Cell 1: Google Search Grounding — Live Web RAG


In [ ]:
response = client.models.generate_content(
    model='gemini-3.6-flash',
    contents='What are the latest developments in India semiconductor policy?',
    config=types.GenerateContentConfig(
        tools=[types.Tool(google_search=types.GoogleSearch())]
    ),
)
print(response.text[:500])


## Cell 2: Extract Grounding Metadata


In [ ]:
gm = response.candidates[0].grounding_metadata

# What the model searched for
print(f'Search queries: {gm.web_search_queries}')

# Source URIs
if gm.grounding_chunks:
    for chunk in gm.grounding_chunks:
        print(f'  Source: {chunk.web.title}')
        print(f'  URI: {chunk.web.uri}')

# Text-to-source mappings
if gm.grounding_supports:
    for support in gm.grounding_supports:
        print(f'  Claim: {support.segment.text[:80]}...')
        print(f'  Backed by: {support.grounding_chunk_indices}')


## Cell 3: Search Widget (Required by ToS)


In [ ]:
from IPython.display import HTML, display

if gm.search_entry_point:
    display(HTML(gm.search_entry_point.rendered_content))
else:
    print('No search widget in this response')


## Cell 4: Create Vertex AI Search Data Store


In [ ]:
# Note: This requires the Discovery Engine API enabled
# gcloud services enable discoveryengine.googleapis.com

ds_client = discoveryengine.DataStoreServiceClient()
parent = ds_client.collection_path(PROJECT_ID, LOCATION, 'default_collection')

data_store = discoveryengine.DataStore(
    display_name='DocuMind KB',
    industry_vertical=discoveryengine.IndustryVertical.GENERIC,
    solution_types=[discoveryengine.SolutionType.SOLUTION_TYPE_SEARCH],
    content_config=discoveryengine.DataStore.ContentConfig.CONTENT_REQUIRED)

try:
    op = ds_client.create_data_store(
        request=discoveryengine.CreateDataStoreRequest(
            parent=parent, data_store_id='documind-kb-test', data_store=data_store))
    result = op.result()
    print(f'Data store: {result.name}')
except Exception as e:
    print(f'Error (may already exist): {e}')


## Cell 5: Import Documents from GCS


In [ ]:
doc_client = discoveryengine.DocumentServiceClient()
branch = (f'projects/{PROJECT_ID}/locations/{LOCATION}'
          f'/collections/default_collection/dataStores/documind-kb-test'
          f'/branches/default_branch')

try:
    import_op = doc_client.import_documents(
        request=discoveryengine.ImportDocumentsRequest(
            parent=branch,
            gcs_source=discoveryengine.GcsSource(
                input_uris=['gs://YOUR-BUCKET/docs/*'],  # CHANGE
                data_schema='content'),
            reconciliation_mode=discoveryengine.ImportDocumentsRequest
                .ReconciliationMode.INCREMENTAL))
    result = import_op.result(timeout=600)
    print('Documents imported')
except Exception as e:
    print(f'Error: {e}')


## Cell 6: Search with Summary


In [ ]:
search_client = discoveryengine.SearchServiceClient()
serving_config = search_client.serving_config_path(
    PROJECT_ID, LOCATION, 'documind-kb-test', 'default_config')

try:
    request = discoveryengine.SearchRequest(
        serving_config=serving_config,
        query='What is RAG?',
        page_size=5,
        content_search_spec=discoveryengine.SearchRequest.ContentSearchSpec(
            snippet_spec=discoveryengine.SearchRequest.ContentSearchSpec
                .SnippetSpec(return_snippet=True),
            summary_spec=discoveryengine.SearchRequest.ContentSearchSpec
                .SummarySpec(summary_result_count=3, include_citations=True)))
    resp = search_client.search(request)
    if resp.summary:
        print(f'Summary: {resp.summary.summary_text}')
    for r in resp.results:
        doc = r.document.derived_struct_data
        print(f'  Title: {doc.get("title")}')
except Exception as e:
    print(f'Error: {e}')


## Cell 7: GroundedSearch Module


In [ ]:
class GroundedSearch:
    def __init__(self, project, location='global'):  # generation-only client -> global
        self.client = genai.Client(enterprise=True, project=project, location=location)
        self.query_count = 0
        self.total_cost = 0.0

    def search(self, question, model='gemini-3.6-flash'):
        response = self.client.models.generate_content(
            model=model, contents=question,
            config=types.GenerateContentConfig(
                tools=[types.Tool(google_search=types.GoogleSearch())]))
        self.query_count += 1
        self.total_cost += 0.035
        citations = []
        gm = response.candidates[0].grounding_metadata
        if gm and gm.grounding_chunks:
            for chunk in gm.grounding_chunks:
                citations.append({'title': chunk.web.title, 'uri': chunk.web.uri})
        return {
            'answer': response.text,
            'citations': citations,
            'queries': gm.web_search_queries if gm else [],
            'widget': gm.search_entry_point.rendered_content if gm and gm.search_entry_point else ''}

    def report(self):
        print(f'Queries: {self.query_count} | Cost: ${self.total_cost:.2f}')

gs = GroundedSearch(PROJECT_ID)
r = gs.search('Latest AI developments in India 2026')
print(f'Answer: {r["answer"][:200]}...')
print(f'Citations: {len(r["citations"])}')
for c in r['citations'][:3]:
    print(f'  {c["title"]}: {c["uri"]}')
gs.report()


## Cell 8: Cost Comparison Calculator


In [ ]:
def compare_costs(queries_per_month):
    q = queries_per_month
    costs = {
        'DIY RAG (Firestore)': q * 0.0003 + 0,  # ~$0.0003/query, no infra
        'RAG Engine': 65 + q * 0.0025,  # $65 base + $2.50/1K grounding
        'Vertex AI Search (Standard)': q * 0.0015 + 5,  # $1.50/1K + storage
        'Vertex AI Search (Enterprise)': q * 0.004 + 5,  # $4/1K + storage
        'Google Search (2.5)': max(0, q - 15000) * 0.035 / 1,  # 500/day free
        'Google Search (3.x)': max(0, q - 150000) * 0.014 / 1,  # 5K/mo free
    }
    print(f'Monthly cost comparison at {q:,} queries/month:')
    for name, cost in sorted(costs.items(), key=lambda x: x[1]):
        inr = cost * 85
        print(f'  {name}: ${cost:.2f} (Rs {inr:.0f})')

compare_costs(10000)
print()
compare_costs(100000)


## ✅ Lesson 4.4 Complete! Module 4 Complete!

**Four RAG approaches mastered:**
- 4.1: Document AI — ingestion layer (OCR/Layout/Form)
- 4.2: DIY RAG — full control pipeline (embed→retrieve→generate)
- 4.3: RAG Engine — managed pipeline (corpus→import→query)
- 4.4: Search + Grounding — enterprise search + live web RAG

**Modules built:** document_ingestion.py, rag_engine.py, managed_rag.py, grounded_search.py

**Next: Module 5 — BigQuery ML & SQL-Native AI**
